In [11]:
import wmfdata as wmf
import pandas as pd
import numpy as np
from wmfdata import spark,hive
import plotly.express as px

In [13]:
# Query

regional_uniques = wmf.spark.run("""
SELECT 
    cmd.wmf_region,
    ud.country_code,
    ud.country,
    AVG(CASE WHEN ud.month IN (7, 8, 9) THEN ud.uniques_estimate ELSE NULL END) AS previous_metric_avg,
    AVG(CASE WHEN ud.month IN (10, 11, 12) THEN ud.uniques_estimate ELSE NULL END) AS current_metric_avg
FROM 
    wmf.unique_devices_per_project_family_monthly ud
JOIN 
    gdi.country_meta_data cmd ON ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.project_family = 'wikipedia'
    AND ud.year = 2023
    AND ud.month IN (7, 8, 9, 10, 11, 12)
GROUP BY 
    cmd.wmf_region,
    ud.country_code,
    ud.country




""")

df = regional_uniques.copy()

# Initialize an empty DataFrame to hold all results
final_df = pd.DataFrame()

# The loop through unique_months is no longer necessary since we're working with aggregated quarterly data

# Directly use the 'df' DataFrame for analysis
df['absolute_change'] = df['current_metric_avg'] - df['previous_metric_avg']

# Calculate Regional and Global Totals
regional_totals = df.groupby('wmf_region')['current_metric_avg', 'previous_metric_avg'].sum().reset_index()
global_current_metric_total = df['current_metric_avg'].sum()
global_previous_metric_total = df['previous_metric_avg'].sum()

# Merge with Regional Totals and add Global Totals
df['global_current_metric_total'] = global_current_metric_total
df['global_previous_metric_total'] = global_previous_metric_total

# Calculating Proportions and Formula Results
df['proportion_current_regional'] = df['current_metric_avg'] / df.groupby('wmf_region')['current_metric_avg'].transform('sum')
df['proportion_previous_regional'] = df['previous_metric_avg'] / df.groupby('wmf_region')['previous_metric_avg'].transform('sum')
df['proportion_current_global'] = df['current_metric_avg'] / global_current_metric_total
df['proportion_previous_global'] = df['previous_metric_avg'] / global_previous_metric_total

df['formula_result_regional'] = abs((df['absolute_change'] * df['proportion_current_regional']) +
                                    ((df['proportion_current_regional'] - df['proportion_previous_regional']) * (df['previous_metric_avg'] - df.groupby('wmf_region')['previous_metric_avg'].transform('sum'))))
df['formula_result_global'] = abs((df['absolute_change'] * df['proportion_current_global']) +
                                  ((df['proportion_current_global'] - df['proportion_previous_global']) * (df['previous_metric_avg'] - global_previous_metric_total)))

# Calculate the Total Sum of 'formula_result' per Region and Globally
df['formula_result_regional_sum'] = df.groupby('wmf_region')['formula_result_regional'].transform('sum')
df['total_formula_result_global'] = df['formula_result_global'].sum()

# Calculate percentages
df['formula_result_percentage_regional'] = (df['formula_result_regional'] / df['formula_result_regional_sum']) * 100
df['formula_result_percentage_global'] = (df['formula_result_global'] / df['total_formula_result_global']) * 100

# Add ranking logic
df['rank_simple_calculation'] = df.groupby('wmf_region')['absolute_change'].rank(method='dense', ascending=False)
df['rank_formula_result'] = df.groupby('wmf_region')['formula_result_percentage_regional'].rank(method='dense', ascending=False)

# Final DataFrame for output
final_df = df

# Sort the DataFrame by 'wmf_region', and 'absolute_change'
final_df = final_df.sort_values(by=['wmf_region', 'absolute_change'], ascending=[True, False])

final_df['month'] = 'Quarter'

final_df = final_df[['month', 'country', 'wmf_region',  'current_metric_avg', 'previous_metric_avg', 'absolute_change', 'proportion_current_regional', 'formula_result_regional', 'proportion_current_global', 'formula_result_global']]

# Save the sorted DataFrame to a CSV file
print("Saving CSV file")
final_df.to_csv("unique_devices_time_series_analysis_quarterly.csv", index=False)

# Show df
final_df

Saving CSV file


/tmp/ipykernel_827/2935049029.py:39: FutureWarning: Indexing with multiple keys (implicitly converted to a tuple of keys) will be deprecated, use a list instead.
  regional_totals = df.groupby('wmf_region')['current_metric_avg', 'previous_metric_avg'].sum().reset_index()


,month,country,wmf_region,current_metric_avg,previous_metric_avg,absolute_change,proportion_current_regional,formula_result_regional,proportion_current_global,formula_result_global
200,Quarter,Russia,Central & Eastern Europe & Central Asia,7.095996e+07,6.103545e+07,9.924502e+06,3.025352e-01,4.983955e+06,4.244836e-02,3.168806e+06
86,Quarter,Turkey,Central & Eastern Europe & Central Asia,2.556568e+07,2.173758e+07,3.828101e+06,1.089984e-01,1.121198e+06,1.529343e-02,1.518473e+06
223,Quarter,Ukraine,Central & Eastern Europe & Central Asia,1.917404e+07,1.605906e+07,3.114981e+06,8.174782e-02,5.766342e+05,1.146994e-02,1.392857e+06
43,Quarter,Poland,Central & Eastern Europe & Central Asia,2.326526e+07,2.055045e+07,2.714814e+06,9.919060e-02,1.600951e+06,1.391731e-02,6.399887e+05
227,Quarter,Kazakhstan,Central & Eastern Europe & Central Asia,8.787097e+06,6.644326e+06,2.142771e+06,3.746347e-02,4.547748e+05,5.256455e-03,1.359633e+06
...,...,...,...,...,...,...,...,...,...,...
19,Quarter,French Southern and Antarctic Lands,Sub-Saharan Africa,2.000000e+00,NaN,NaN,5.815793e-08,NaN,1.196403e-09,NaN
40,Quarter,"Saint Helena, Ascension, and Tristan da Cunha",Sub-Saharan Africa,1.217500e+03,NaN,NaN,3.540364e-05,NaN,7.283104e-07,NaN
41,Quarter,Republic of the Congo,Sub-Saharan Africa,1.351300e+05,NaN,NaN,3.929441e-03,NaN,8.083498e-05,NaN
249,Quarter,Cape Verde,Sub-Saharan Africa,6.858750e+04,NaN,NaN,1.994454e-03,NaN,4.102915e-05,NaN


## Mobile Wiki breakdown

In [ ]:
# Query

regional_uniques_mobile = wmf.spark.run("""
SELECT 
    cmd.wmf_region,
    ud.country_code,
    ud.country,
    AVG(CASE WHEN ud.month IN (7, 8, 9) THEN ud.uniques_estimate ELSE NULL END) AS previous_metric,
    AVG(CASE WHEN ud.month IN (10, 11, 12) THEN ud.uniques_estimate ELSE NULL END) AS current_metric
FROM 
    wmf.unique_devices_per_domain_monthly ud
JOIN 
    gdi.country_meta_data cmd ON ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.domain LIKE '%.m.wikipedia%'
    AND ud.year = 2023
GROUP BY 
    cmd.wmf_region,
    ud.country_code,
    ud.country


""")

df = regional_uniques_mobile.copy()

# Calculate Absolute Change directly
df['absolute_change'] = df['current_metric'] - df['previous_metric']

# Calculate Regional and Global Totals
regional_totals = df.groupby('wmf_region')['current_metric', 'previous_metric'].sum().reset_index()
global_current_metric_total = df['current_metric'].sum()
global_previous_metric_total = df['previous_metric'].sum()

# Add Global Totals to df for proportion calculations
df['global_current_metric_total'] = global_current_metric_total
df['global_previous_metric_total'] = global_previous_metric_total

# Calculating Proportions and Formula Results
df['proportion_current_regional'] = df['current_metric'] / df.groupby('wmf_region')['current_metric'].transform('sum')
df['proportion_previous_regional'] = df['previous_metric'] / df.groupby('wmf_region')['previous_metric'].transform('sum')
df['proportion_current_global'] = df['current_metric'] / global_current_metric_total
df['proportion_previous_global'] = df['previous_metric'] / global_previous_metric_total

df['formula_result_regional'] = abs((df['absolute_change'] * df['proportion_current_regional']) +
                                    ((df['proportion_current_regional'] - df['proportion_previous_regional']) * (df['previous_metric'] - df.groupby('wmf_region')['previous_metric'].transform('sum'))))
df['formula_result_global'] = abs((df['absolute_change'] * df['proportion_current_global']) +
                                  ((df['proportion_current_global'] - df['proportion_previous_global']) * (df['previous_metric'] - global_previous_metric_total)))

# Calculate the Total Sum of 'formula_result' per Region and Globally
df['formula_result_regional_sum'] = df.groupby('wmf_region')['formula_result_regional'].transform('sum')
df['total_formula_result_global'] = df['formula_result_global'].sum()

# Calculate percentages
df['formula_result_percentage_regional'] = (df['formula_result_regional'] / df['formula_result_regional_sum']) * 100
df['formula_result_percentage_global'] = (df['formula_result_global'] / df['total_formula_result_global']) * 100

# Add ranking logic
df['rank_simple_calculation'] = df.groupby('wmf_region')['absolute_change'].rank(method='dense', ascending=False)
df['rank_formula_result'] = df.groupby('wmf_region')['formula_result_percentage_regional'].rank(method='dense', ascending=False)

# Sort the DataFrame by 'wmf_region', and 'absolute_change'
final_df = df.sort_values(by=['wmf_region', 'absolute_change'], ascending=[True, False])


final_df['month'] = 'Quarter'
final_df = final_df[['month', 'country', 'wmf_region',  'current_metric', 'previous_metric', 'absolute_change', 'proportion_current_regional', 'formula_result_regional', 'proportion_current_global', 'formula_result_global']]

# Saving the DataFrame to a CSV file
print("Saving CSV file")
final_df.to_csv("unique_devices_time_series_analysis_mobile_quarterly.csv", index=False)

# Display the DataFrame
final_df